In [18]:
%load_ext autoreload
%autoreload 2
%matplotlib widget
%pwd

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


'/projectnb/batmanlab/mragoza/lung-project/notebooks/copdgene'

In [19]:
import sys, os

def add_path(p: str):
    if p not in sys.path:
        sys.path.append(p)

add_path(os.environ['LP_ROOT'])
import project

add_path(os.environ['PROJECT'] + '/' + 'param_search')
import param_search as ps

ps.set_backend('sge')
ps.set_verbose(False)

# Gather examples

In [20]:
from pathlib import Path
#data_root = Path(os.environ['LP_ROOT'] / 'data' / 'COPDGene'
data_root = Path(os.environ['PRIVATE']) / 'data' / 'COPDGene'
data_root.is_dir()

True

In [21]:
import pandas as pd
subject_file = data_root / 'sample1000_2025-07-22.csv'
subject_list = list(pd.read_csv(subject_file, sep='\t').sid)
len(subject_list)

1000

In [22]:
bad_subjects = ['23686J', '24634V']
good_subjects = [s for s in subject_list if s not in bad_subjects]
len(good_subjects)

998

# Setup experiment

In [23]:
base_dir = '2026-10-22_train'

template = '''\
#!/bin/bash -l
#$ -N {job_name}
#$ -P batmanlab
#$ -pe omp 4
#$ -l gpus=1
#$ -l gpu_memory=32G
#$ -l h_rt=24:00:00
set -eo pipefail

mamba activate $PROJECT/mambaforge/envs/warp

export PYTHONPATH=$LP_ROOT:$PROJECT:$PYTHOPATH

python $LP_ROOT/scripts/train.py {config} \\
    --set dataset.subjects={subjects} \\
    --set dataset.variant={variant} \\
    --set dataset.pipeline_tags.mesh_generation=rad \\
    --set dataset.pipeline_tags.field_interpolation=fix \\
    --set dataset.pipeline_tags.forward_simulation={forward_tag} \\
    --set training.seed={random_seed}
    --set training.model.backbone.out_channels={conv_channels} \\
    --set training.optimizer.learning_rate={learning_rate} \\
    --set training.pde_solver.material_type={material_type} \\
    --set training.pde_solver.tv_reg_weight={tv_reg_weight}

'''
name_format = 'job_{params_hash}'

grid = ps.param_grid(
    config='2026-09-15_config.yaml',
    variant='2026-09-15',
    subjects=','.join(good_subjects),
    forward_tag=['LE'],
    random_seed=range(3),
    conv_channels=[8, 16],
    learning_rate=[1e-4, 3e-4],
    material_type=['LE'],
    tv_reg_weight=[1e-4]
)

len(grid)

12

In [24]:
jobs = ps.setup(base_dir, template, name_format, grid, overwrite=False)
jobs

OSError: /projectnb/batmanlab/mragoza/lung-project/notebooks/copdgene/2026-10-22_train/job_3360b40d439e34b7/run.sh already exists

In [25]:
#jobs.loc[:, 'job_id'] = pd.NA
jobs = ps.submit(jobs)

# Monitor submitted jobs

In [10]:
jobs = ps.load(base_dir)
jobs

,job_name,job_state,n_submits,job_id,node_id,runtime,stdout,stderr,base_dir,work_dir,...,params.material_type,params.tv_reg_weight,array_idx,last_live_at,state_source,finalized,finalized_at,output_exists,output_fsize,output_mtime
0,job_3360b40d439e34b7,PENDING,1,8247977,None,None,None,None,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,LE,0.0001,NaN,2026-10-22T11:26:34,status,None,None,False,None,None
1,job_41e3893362d2d0ee,PENDING,1,8247978,None,None,None,None,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,LE,0.0001,NaN,2026-10-22T11:26:34,status,None,None,False,None,None
2,job_58b471da6344edb9,PENDING,1,8247979,None,None,None,None,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,LE,0.0001,NaN,2026-10-22T11:26:34,status,None,None,False,None,None
3,job_30233d48f6f4b19f,PENDING,1,8247980,None,None,None,None,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,LE,0.0001,NaN,2026-10-22T11:26:34,status,None,None,False,None,None
4,job_6b7366e0069665a4,PENDING,1,8247981,None,None,None,None,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,LE,0.0001,NaN,2026-10-22T11:26:34,status,None,None,False,None,None
5,job_ccffffe92ad6e372,PENDING,1,8247982,None,None,None,None,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,LE,0.0001,NaN,2026-10-22T11:26:34,status,None,None,False,None,None
6,job_d06fc90622b7231d,PENDING,1,8247983,None,None,None,None,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,LE,0.0001,NaN,2026-10-22T11:26:34,status,None,None,False,None,None
7,job_4b6455413814f36c,PENDING,1,8247984,None,None,None,None,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,LE,0.0001,NaN,2026-10-22T11:26:34,status,None,None,False,None,None
8,job_e5eb5a7b8c87f4ef,PENDING,1,8247985,None,None,None,None,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,LE,0.0001,NaN,2026-10-22T11:26:34,status,None,None,False,None,None
9,job_15cd6ca83a0a8275,PENDING,1,8247986,None,None,None,None,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,LE,0.0001,NaN,2026-10-22T11:26:34,status,None,None,False,None,None


In [12]:
%autoreload
jobs = ps.recover(jobs)
jobs = ps.status(jobs)
jobs = ps.history(jobs) # not working for SGE backend; job_state may be stale
jobs = ps.collect(jobs)

jobs.groupby('job_state').job_name.describe()

,count,unique,top,freq
job_state,,,,
PENDING,12,12,job_3360b40d439e34b7,1


In [15]:
def classify_output(stdout: str):
    if pd.isnull(stdout):
        return 'No stdout file'
    elif not str(stdout).strip():
        return 'No output message'
    elif 'Done' in stdout:
        return 'Complete'
    return 'Incomplete'

def classify_error(stderr: str):
    if pd.isnull(stderr):
        return 'No stderr file'    
    elif not str(stderr).strip():
        return 'No error message'
    elif 'Segmentation fault' in stderr:
        return 'Segmentation fault'
    elif 'CUDA out of memory' in stderr:
        return 'CUDA out of memory'
    elif 'No CUDA GPUs are available' in stderr:
        return 'CUDA not available'
    elif 'Warp CUDA error' in stderr:
        return 'Warp CUDA error'
    elif 'Warp UserWarning' in stderr:
        return 'Warp user warning'
    elif 'Newton solver failed to converge' in stderr:
        return 'Failed to converge'
    elif 'Non-finite PDE residual' in stderr:
        return 'Non-finite PDE residual'
    return 'Uncategorized'

jobs['error_type'] = jobs.stderr.map(classify_error)
jobs['output_type'] = jobs.stdout.map(classify_output)

jobs.groupby(['output_type', 'error_type', ])[['job_name']].count()

,,job_name
output_type,error_type,
Incomplete,Uncategorized,12


In [17]:
query = (jobs.error_type == 'Uncategorized')
query_jobs = jobs[query]

job = query_jobs.iloc[0]
print(job)
print(job.stderr)

job_name                                             job_3360b40d439e34b7
job_state                                                         PENDING
n_submits                                                               1
job_id                                                            8247977
node_id                                                              None
runtime                                                              None
stdout                  [Epoch 0 | val batch 93 | load data]  dt = 4.3...
stderr                      project.api.run_training(examples, config[...
base_dir                /projectnb/batmanlab/mragoza/lung-project/note...
work_dir                /projectnb/batmanlab/mragoza/lung-project/note...
script_path             /projectnb/batmanlab/mragoza/lung-project/note...
output_path             /projectnb/batmanlab/mragoza/lung-project/note...
log_dir                 /projectnb/batmanlab/mragoza/lung-project/note...
stdout_path             /projectnb/bat